In [5]:
from PIL import Image
import requests
from transformers import CLIPModel, CLIPProcessor
import torch

device = "mps" if torch.backends.mps.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

# swap in any direct image URL
url = "/Users/prathi/Downloads/Horses.jpg"  
image = Image.open(url)



Loading weights: 100%|██████████| 398/398 [00:00<00:00, 51394.14it/s]


In [6]:
text = "a photo of three horses running in a field"  

inputs = processor(text=[text], images=image, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds

image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

cosine_sim = (image_embeds @ text_embeds.T).item()
print(f"Cosine similarity: {cosine_sim:.4f}")

Cosine similarity: 0.3017


In [9]:
texts = [
    "a photo of three horses running in a field",
    "a photo of a dog",
    "a photo of horses grazing in a meadow",
    "a photo of a horse",
    "a photo of a city street"
]

inputs = processor(text=texts, images=image, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    image_embeds = outputs.image_embeds
    text_embeds = outputs.text_embeds

image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

sims = (image_embeds @ text_embeds.T).squeeze(0)
for t, s in zip(texts, sims):
    print(f"{t}: {s.item():.4f}")

a photo of three horses running in a field: 0.3017
a photo of a dog: 0.2053
a photo of horses grazing in a meadow: 0.3106
a photo of a horse: 0.2654
a photo of a city street: 0.1753
